# Debugging LangGraph Agents with LangSmith

Configure LangSmith tracing, build tool-calling agents, and inspect trace runs in LangSmith and LangGraph Studio.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY", os.getenv("LANGCHAIN_API_KEY", ""))
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "debugging-agent"

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages
from langchain.chat_models import init_chat_model

class State(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

llm = init_chat_model("google_genai:gemini-2.5-flash")
llm

In [ ]:
from langchain_core.tools import tool

@tool
def add(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b

tools = [add]
llm_with_tool = llm.bind_tools(tools)

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition

def call_llm_model(state: State):
    return {"messages": [llm_with_tool.invoke(state["messages"])]}

builder = StateGraph(State)
builder.add_node("tool_calling_llm", call_llm_model)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "tool_calling_llm")
builder.add_conditional_edges("tool_calling_llm", tools_condition)
builder.add_edge("tools", "tool_calling_llm")

graph = builder.compile()

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    pass

In [ ]:
# 1. Non-tool call invocation (traced in LangSmith)
response = graph.invoke({"messages": "What is machine learning?"})
print(response["messages"][-1].content)

In [ ]:
# 2. Tool call invocation (traced in LangSmith)
response = graph.invoke({"messages": "What is 5 plus 20?"})
for m in response["messages"]:
    m.pretty_print()